Bibliotecas que serão utilizadas

In [17]:
import pandas as pd
import requests
import os
import time
import datetime

Criando as pastas necessarias para o projeto

In [18]:
os.makedirs("Base CNPJs", exist_ok=True)

os.makedirs("Dados Prontos", exist_ok=True)

os.makedirs("Erros", exist_ok=True)

pastas_do_projeto = os.listdir()

for pasta in pastas_do_projeto:

    if os.path.isdir(pasta):

        print(pasta)

.git
Base CNPJs
Dados Prontos
Erros


Caminho para as pastas do projeto

In [19]:
Lista_Base_CNPJs = os.listdir("Base CNPJs")

Dados_Prontos = "Dados Prontos"

Erros = "Erros"

Realizando requisições a API da ReceitaWS

In [20]:

lista_dados = []

for arquivo in Lista_Base_CNPJs:

    if arquivo.endswith(".xlsx"):

        try:

            dados_cnpj = pd.read_excel(f"Base CNPJs/{arquivo}", header=None)

            dados_cnpj = dados_cnpj.dropna(how="all")

            dados_cnpj.columns = ["CNPJ"]

            dados_cnpj = dados_cnpj[dados_cnpj["CNPJ"] != "CNPJ"]

            for cnpj in dados_cnpj["CNPJ"]:

                time.sleep(20)

                cnpj_limpo = (str(cnpj).strip().replace(".","").replace("-","").replace("/",""))
                
                requisicao = requests.get(f"https://receitaws.com.br/v1/cnpj/{cnpj_limpo}")

                if requisicao.status_code != 200:
                    
                    print(f"Falha na busca pelo CNPJ {cnpj_limpo} da planilha {arquivo}. Código: {requisicao.status_code}")

                    with open(f"{Erros}/logs.txt", "a", encoding="utf-8") as log:

                        log.write(f"Falha na busca pelo CNPJ {cnpj_limpo} da planilha {arquivo}. código: {requisicao.status_code}\n")

                        data_requisicao = datetime.datetime.now().strftime("%d/%m/%y %H:%M")

                        log.write(f"Data da Requisição: {data_requisicao}\n")

                        log.write("---------------------------------------------------------------------------------------------------\n")
                
                else:
                    
                    lista_dados.append(requisicao.json())

                    print(f"O CNPJ {cnpj_limpo} da planilha {arquivo} foi encontrado")
                    


        except requests.exceptions.JSONDecodeError:

            print(f"A API não retornou os dados no formato json. Ela retornou {requisicao.text}\n")

            with open(f"{Erros}/logs.txt", "a", encoding="utf-8") as log:

                log.write(f"A API não retornou os dados no formato json. Ela retornou {requisicao.text}\n")

                data_requisicao = datetime.datetime.now().strftime("%d/%m/%y %H:%M")

                log.write(f"Data da Requisição: {data_requisicao}\n")

                log.write("-------------------------------------------------------------------------\n")
        
        except requests.exceptions.ConnectionError as erro:

            print("O limite de requisição a API foi alcançado")

            with open(f"{Erros}/logs.txt", "a", encoding="utf-8") as log:

                log.write(f"O limite de requisições a API foi alcançado. Código: {erro}\n")

                log.write(f"Código: {requisicao.status_code}\n")

                data_requisicao = datetime.datetime.now().strftime("%d/%m/%y %H:%M")

                log.write(f"Data da Requisição: {data_requisicao}\n")

                log.write("-------------------------------------------------------------------------\n")
                
    else:

        print(f"O arquivo {arquivo} nao é um excel")

        with open(f"{Erros}/logs.txt", "a", encoding="utf-8") as log:

            log.write(f"O arquivo {arquivo} não é um excel\n")

            data_requisicao = datetime.datetime.now().strftime("%d/%m/%y %H:%M")

            log.write(f"Data da Requisição: {data_requisicao}\n")

            log.write("-------------------------------------------------------------------------\n")


KeyboardInterrupt: 

Visualizando a lista de dicionários JSON gerada

In [ ]:
lista_dados

[{'abertura': '20/11/2000',
  'situacao': 'ATIVA',
  'tipo': 'MATRIZ',
  'nome': 'M&C IMPERIAL MOLAS & SUSPENSAO LTDA',
  'fantasia': 'M&C IMPERIAL MOLAS & SUSPENSAO',
  'porte': 'EMPRESA DE PEQUENO PORTE',
  'natureza_juridica': '206-2 - Sociedade Empresária Limitada',
  'atividade_principal': [{'code': '45.20-0-01',
    'text': 'Serviços de manutenção e reparação mecânica de veículos automotores'}],
  'atividades_secundarias': [{'code': '23.30-3-01',
    'text': 'Fabricação de estruturas pré-moldadas de concreto armado, em série e sob encomenda'},
   {'code': '23.30-3-02',
    'text': 'Fabricação de artefatos de cimento para uso na construção'},
   {'code': '25.39-0-01', 'text': 'Serviços de usinagem, torneiria e solda'},
   {'code': '25.42-0-00',
    'text': 'Fabricação de artigos de serralheria, exceto esquadrias'},
   {'code': '42.11-1-01', 'text': 'Construção de rodovias e ferrovias'},
   {'code': '42.13-8-00',
    'text': 'Obras de urbanização - ruas, praças e calçadas'},
   {'c

Criando o Dataframe usando a lista de dados

In [ ]:
base_de_dados_empresas = pd.DataFrame(lista_dados)

Visualizando as 5 primeiras linhas

In [ ]:
pd.set_option('display.max_columns', None)

base_de_dados_empresas.head()

,abertura,situacao,tipo,nome,fantasia,porte,natureza_juridica,atividade_principal,atividades_secundarias,qsa,logradouro,numero,complemento,municipio,bairro,uf,cep,email,telefone,data_situacao,cnpj,ultima_atualizacao,status,efr,motivo_situacao,situacao_especial,data_situacao_especial,capital_social,simples,simei,extra,billing
0,20/11/2000,ATIVA,MATRIZ,M&C IMPERIAL MOLAS & SUSPENSAO LTDA,M&C IMPERIAL MOLAS & SUSPENSAO,EMPRESA DE PEQUENO PORTE,206-2 - Sociedade Empresária Limitada,"[{'code': '45.20-0-01', 'text': 'Serviços de m...","[{'code': '23.30-3-01', 'text': 'Fabricação de...","[{'nome': 'ALESSANDRO RODRIGUES DA SILVA', 'qu...",RUA PROGRESSO,S/N,QUADRA048 SITIO SANTO ANTONIO,CANAA DOS CARAJAS,POLO INDUSTRIAL,PA,68.350-345,sandro.mcimperial@gmail.com,(94) 9211-4534,15/12/2020,04.158.876/0001-11,2026-08-08T23:59:59.000Z,OK,,,,,1000000.00,"{'optante': False, 'data_opcao': '01/01/2021',...","{'optante': False, 'data_opcao': None, 'data_e...",{},"{'free': True, 'database': True}"
1,29/09/2000,ATIVA,MATRIZ,PROMAQUINAS TERRAPLANAGEM E TELEFONIA LTDA,PROMAQUINAS SERVICOS E EQUIPAMENTOS,EMPRESA DE PEQUENO PORTE,206-2 - Sociedade Empresária Limitada,"[{'code': '61.90-6-99', 'text': 'Outras ativid...","[{'code': '08.10-0-06', 'text': 'Extração de a...","[{'nome': 'ANNA MEL PASSOS DE ALMEIDA', 'qual'...",RODOVIA BR 101,155,,ITABUNA,SAO LOURENCO,BA,45.602-672,agape.contabilidade@hotmail.com,(73) 3212-4074,24/07/2008,04.077.298/0001-99,2026-08-08T23:59:59.000Z,OK,,,,,1000000.00,"{'optante': True, 'data_opcao': '01/01/2024', ...","{'optante': False, 'data_opcao': None, 'data_e...",{},"{'free': True, 'database': True}"
2,14/06/2000,ATIVA,MATRIZ,COLLINA HOTEL LTDA,COLLINA HOTEL,MICRO EMPRESA,206-2 - Sociedade Empresária Limitada,"[{'code': '55.10-8-01', 'text': 'Hotéis'}]","[{'code': '42.11-1-01', 'text': 'Construção de...","[{'nome': 'MARCELO MOCELLIN', 'qual': '49-Sóci...",RUA PEDRO CELESTINO,2050,,CAMAPUA,VL JARDIM AMERICA,MS,79.420-000,financeiro.collinahotel@hotmail.com,(67) 3286-1593 / (67) 9962-0660 / (067) 2861-333,12/04/2003,03.886.056/0001-83,2026-08-08T23:59:59.000Z,OK,,,,,2770000.00,"{'optante': True, 'data_opcao': '01/01/2020', ...","{'optante': False, 'data_opcao': None, 'data_e...",{},"{'free': True, 'database': True}"
3,15/12/2000,ATIVA,MATRIZ,EDUARDO RONDINA METALURGICA,METALURGICA METALRON,MICRO EMPRESA,213-5 - Empresário (Individual),"[{'code': '24.51-2-00', 'text': 'Fundição de f...","[{'code': '33.14-7-11', 'text': 'Manutenção e ...",[],RUA DR. IVO BERTI,180,,ATIBAIA,DO TANQUE,SP,12.953-162,ocof2000@ig.com.br,(011) 4032-4797 / (011) 4032-4797,16/07/2005,04.210.899/0001-28,2026-08-08T23:59:59.000Z,OK,,,,,0.00,"{'optante': True, 'data_opcao': '01/07/2007', ...","{'optante': False, 'data_opcao': None, 'data_e...",{},"{'free': True, 'database': True}"
4,09/06/2000,ATIVA,MATRIZ,NB METALURGICA LTDA,,MICRO EMPRESA,206-2 - Sociedade Empresária Limitada,"[{'code': '24.51-2-00', 'text': 'Fundição de f...","[{'code': '00.00-0-00', 'text': 'Não informada'}]","[{'nome': 'RAFAEL GARCIA', 'qual': '22-Sócio'}...",AVENIDA MARACANA,6000,,ARAPONGAS,PARQUE INDUSTRIAL,PR,86.703-000,metalurgicaaguias@hotmail.com,(43) 3276-2191,12/03/2005,03.868.507/0001-50,2026-08-08T23:59:59.000Z,OK,,,,,20000.00,"{'optante': True, 'data_opcao': '01/07/2007', ...","{'optante': False, 'data_opcao': None, 'data_e...",{},"{'free': True, 'database': True}"


Visualizando as colunas do DataFrame: Nessa etapa, vamos definir quais colunas são necessárias para a nossa análise. Essa escolha será baseada no que definimos no escopo de construção do dashboard do power bi

In [ ]:
base_de_dados_empresas.columns

Index(['abertura', 'situacao', 'tipo', 'nome', 'fantasia', 'porte',
       'natureza_juridica', 'atividade_principal', 'atividades_secundarias',
       'qsa', 'logradouro', 'numero', 'complemento', 'municipio', 'bairro',
       'uf', 'cep', 'email', 'telefone', 'data_situacao', 'cnpj',
       'ultima_atualizacao', 'status', 'efr', 'motivo_situacao',
       'situacao_especial', 'data_situacao_especial', 'capital_social',
       'simples', 'simei', 'extra', 'billing'],
      dtype='object')

Colunas Escolhidas para a análise: cnpj, nome, situacao, municipio, uf, email, telefone, atividade_principal, porte.

Nessa etapa vamos excluir as colunas que não iremos utilizar no nosso projeto 

In [ ]:
base_de_dados_empresas = base_de_dados_empresas.drop(columns=["abertura", "tipo","fantasia","natureza_juridica", "atividades_secundarias","qsa","logradouro","numero", "complemento", "bairro", "cep", "data_situacao", "ultima_atualizacao", "status","efr","motivo_situacao","situacao_especial", "data_situacao_especial","capital_social", "simples", "simei","extra", "billing"])

Após a exclusão vamos verificar as informações principais do nosso dataframe filtrado

In [ ]:
base_de_dados_empresas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   situacao             11 non-null     object
 1   nome                 11 non-null     object
 2   porte                11 non-null     object
 3   atividade_principal  11 non-null     object
 4   municipio            11 non-null     object
 5   uf                   11 non-null     object
 6   email                11 non-null     object
 7   telefone             11 non-null     object
 8   cnpj                 11 non-null     object
dtypes: object(9)
memory usage: 924.0+ bytes


Verificando os valores unicos de cada coluna

In [ ]:
for coluna in base_de_dados_empresas:

    valores_unicos = base_de_dados_empresas[coluna].value_counts()

    print(valores_unicos)

    print("------------------------------------------------------------")

situacao
ATIVA    11
Name: count, dtype: int64
------------------------------------------------------------
nome
M&C IMPERIAL MOLAS & SUSPENSAO LTDA                     1
PROMAQUINAS TERRAPLANAGEM E TELEFONIA LTDA              1
COLLINA HOTEL LTDA                                      1
EDUARDO RONDINA METALURGICA                             1
NB METALURGICA LTDA                                     1
CONSTRUTORA MILENIO LTDA                                1
PAVICAMPOS INDUSTRIA E SERVICOS DE PAVIMENTACAO LTDA    1
F EDILSON C CORREIA & CIA LTDA                          1
A H R XAVIER LTDA                                       1
QUEMULIANE ENGENHARIA E SOLUCOES LTDA                   1
L2 ENGENHARIA LTDA                                      1
Name: count, dtype: int64
------------------------------------------------------------
porte
MICRO EMPRESA               7
EMPRESA DE PEQUENO PORTE    3
DEMAIS                      1
Name: count, dtype: int64
----------------------------------------

Nessa etapa vamos verificar os dados de cada coluna com o objetivo de garantir que os formatos de cada dado esta correto

In [ ]:
base_de_dados_empresas['situacao']

0     ATIVA
1     ATIVA
2     ATIVA
3     ATIVA
4     ATIVA
5     ATIVA
6     ATIVA
7     ATIVA
8     ATIVA
9     ATIVA
10    ATIVA
Name: situacao, dtype: object

In [ ]:
base_de_dados_empresas['nome']

0                   M&C IMPERIAL MOLAS & SUSPENSAO LTDA
1            PROMAQUINAS TERRAPLANAGEM E TELEFONIA LTDA
2                                    COLLINA HOTEL LTDA
3                           EDUARDO RONDINA METALURGICA
4                                   NB METALURGICA LTDA
5                              CONSTRUTORA MILENIO LTDA
6     PAVICAMPOS INDUSTRIA E SERVICOS DE PAVIMENTACA...
7                        F EDILSON C CORREIA & CIA LTDA
8                                     A H R XAVIER LTDA
9                 QUEMULIANE ENGENHARIA E SOLUCOES LTDA
10                                   L2 ENGENHARIA LTDA
Name: nome, dtype: object

In [ ]:
base_de_dados_empresas['porte']

0     EMPRESA DE PEQUENO PORTE
1     EMPRESA DE PEQUENO PORTE
2                MICRO EMPRESA
3                MICRO EMPRESA
4                MICRO EMPRESA
5                       DEMAIS
6                MICRO EMPRESA
7                MICRO EMPRESA
8                MICRO EMPRESA
9                MICRO EMPRESA
10    EMPRESA DE PEQUENO PORTE
Name: porte, dtype: object

In [ ]:
base_de_dados_empresas['atividade_principal']

0     [{'code': '45.20-0-01', 'text': 'Serviços de m...
1     [{'code': '61.90-6-99', 'text': 'Outras ativid...
2            [{'code': '55.10-8-01', 'text': 'Hotéis'}]
3     [{'code': '24.51-2-00', 'text': 'Fundição de f...
4     [{'code': '24.51-2-00', 'text': 'Fundição de f...
5     [{'code': '41.20-4-00', 'text': 'Construção de...
6     [{'code': '77.31-4-00', 'text': 'Aluguel de má...
7     [{'code': '41.20-4-00', 'text': 'Construção de...
8     [{'code': '47.44-0-05', 'text': 'Comércio vare...
9     [{'code': '41.20-4-00', 'text': 'Construção de...
10    [{'code': '41.20-4-00', 'text': 'Construção de...
Name: atividade_principal, dtype: object

In [ ]:
base_de_dados_empresas['municipio']

0            CANAA DOS CARAJAS
1                      ITABUNA
2                      CAMAPUA
3                      ATIBAIA
4                    ARAPONGAS
5                SANTO ESTEVAO
6        CAMPOS DOS GOYTACAZES
7                       CAXIAS
8     DOIS IRMAOS DO TOCANTINS
9               RIO DE JANEIRO
10                   FORTALEZA
Name: municipio, dtype: object

In [ ]:
base_de_dados_empresas['uf']

0     PA
1     BA
2     MS
3     SP
4     PR
5     BA
6     RJ
7     MA
8     TO
9     RJ
10    CE
Name: uf, dtype: object

In [ ]:
base_de_dados_empresas['email']

0             sandro.mcimperial@gmail.com
1         agape.contabilidade@hotmail.com
2     financeiro.collinahotel@hotmail.com
3                      ocof2000@ig.com.br
4           metalurgicaaguias@hotmail.com
5         washingtonluisdasilva@gmail.com
6         jgm@construsanengenharia.com.br
7             fedilsoncorreia@hotmail.com
8                   mineracaojx@gmail.com
9               quimoelsoares02@gmail.com
10          luizlopes@l2engenharia.com.br
Name: email, dtype: object

In [ ]:
base_de_dados_empresas['telefone']

0                                       (94) 9211-4534
1                                       (73) 3212-4074
2     (67) 3286-1593 / (67) 9962-0660 / (067) 2861-333
3                    (011) 4032-4797 / (011) 4032-4797
4                                       (43) 3276-2191
5                                       (71) 8655-6321
6                                       (22) 2738-5445
7                                       (99) 3521-5761
8                                       (63) 8435-3216
9                                       (21) 3936-5351
10                      (85) 9981-9449 / (85) 2444-445
Name: telefone, dtype: object

In [ ]:
base_de_dados_empresas['cnpj']

0     04.158.876/0001-11
1     04.077.298/0001-99
2     03.886.056/0001-83
3     04.210.899/0001-28
4     03.868.507/0001-50
5     04.077.304/0001-08
6     04.014.959/0001-37
7     04.012.485/0001-94
8     03.859.533/0001-11
9     03.711.353/0001-98
10    03.751.025/0001-15
Name: cnpj, dtype: object

Após revisar os valores, percebemos que será necessário ajustar os valores de duas colunas (atividade principal e telefone)

Ajustando a atividade principal: Basicamente essa coluna retorna um dicionário que possui um código identificador e um texto que descreve o serviço prestado
pela empresa. Dito isso, será necessário filtrar apenas o texto que descreve o serviço, pois, o código não será necessário no contexto desse projeto.

In [ ]:
base_de_dados_empresas['atividade_principal'] = base_de_dados_empresas['atividade_principal'].str[0].str['text']

Verificando o resultado

In [ ]:
base_de_dados_empresas['atividade_principal']

0     Serviços de manutenção e reparação mecânica de...
1     Outras atividades de telecomunicações não espe...
2                                                Hotéis
3                               Fundição de ferro e aço
4                               Fundição de ferro e aço
5                               Construção de edifícios
6     Aluguel de máquinas e equipamentos agrícolas s...
7                               Construção de edifícios
8     Comércio varejista de materiais de construção ...
9                               Construção de edifícios
10                              Construção de edifícios
Name: atividade_principal, dtype: object

Ajustando a coluna de telefone: Basicamente vamos apagar os valores duplicados de telefone que o ReceitaWS retornou

In [ ]:
base_de_dados_empresas['telefone'] = base_de_dados_empresas['telefone'].str.split('/').str[0]

In [ ]:
base_de_dados_empresas['telefone']

0       (94) 9211-4534
1       (73) 3212-4074
2      (67) 3286-1593 
3     (011) 4032-4797 
4       (43) 3276-2191
5       (71) 8655-6321
6       (22) 2738-5445
7       (99) 3521-5761
8       (63) 8435-3216
9       (21) 3936-5351
10     (85) 9981-9449 
Name: telefone, dtype: object

In [ ]:
data_arquivo = datetime.datetime.now().strftime("%d/%m/%y %H:%M")

base_de_dados_empresas.to_excel(f"{Dados_Prontos}/Relatório_{data_arquivo}")

ValueError: No engine for filetype: ''